# Automatic Gradient

## Author: RAHUL KP KURUP

This notebook implemtes a minimal automatic differentiation engine and builds a neural network (MLP) from scratch.

# Features
- Scalar-based autograd engine
- Backpropagation using topological sort
- Activation Function: tanh, ReLu, exp
- Neural network: Neuron -> Layer -> MLP
- Visualization support (Graphviz)

# 1. Core Autograd Engine

# Object Creation

In [21]:
import math

class Value:

    def __init__(self, data, children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = _op
        self.label = label

v = Value(5)
print(v)
v.grad = 2
v._op = '+'
v.label = '+'
print(v.data, v.grad, v.label, v._op)

5.0 2 + +


# "__repr__" function

In [1]:
import math
class sample:
    def __init__(self, data):
        self.data = data
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

v = sample(5)
print(v)

Value(data=5, grad=0)


# Adition function

## Gradient of Addition

We want to compute the derivative:

$$\frac{d}{dx}(x + x) = ?$$

### Step-by-step

Simplify the expression:

$$x + x = 2x$$

Now differentiate:

$$\frac{d}{dx}(2x) = 2$$

### Final Answer

$$\frac{d}{dx}(x + x) = 2$$

###  Intuition

- Each variable contributes a gradient of **1**
- So:
  - First x → 1
  - Second x → 1

 Total:

$$1 + 1 = 2$$

In [10]:
# Simple numerical verification
import numpy as np

def f(x):
    return x + x

x = 5
h = 1e-5

# Numerical derivative 
grad = (f(x + h) - f(x - h)) / (2 * h)

print('Numerical Gradient:', grad)

Numerical Gradient: 1.9999999999242843


In [6]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._children = _children   # computation graph
        self._op = _op               # operation
        self._backward = lambda: None  # default empty function

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        # Ensure both are Value objects
        other = other if isinstance(other, Value) else Value(other)
        
        # Create output node
        out = Value(self.data + other.data, (self, other), '+')

        # Define backward function (INSIDE)
        def _backward():
        # Gradient rule:
        # d/dx (x + x) = 2
        # More generally:
        # d/dself (self + other) = 1
        # d/dother (self + other) = 1
        # So each input receives the full upstream gradient (out.grad)
            self.grad += out.grad
            other.grad += out.grad

        # Attach backward function to output node
        out._backward = _backward

        return out
v1 = Value(10)
# v2 = Value(5)
v2 = 10
result = v1 + v2
print(result)
    

Value(data=20, grad=0)


# Multiplication Function

## Gradient of Multiplication

We want to compute the derivative:

$$\frac{d}{dx}(x \cdot x) = ?$$

### Step-by-step

Simplify the expression:

$$x \cdot x = x^2$$

### Final Answer

$$\frac{d}{dx}(x \cdot x) = 2x$$

### Intuition

- Each variable contributes the other variable
- So:
  - First x → contributes x
  - Second x → contributes x

Total:

$$x + x = 2x$$

In [11]:
import numpy as np

def f(x):
    return x * x

x = 5
h = 1e-5

# Numerical derivative
grad = (f(x + h) - f(x - h)) / (2 * h)

print('Numerical Gradient:', grad)

Numerical Gradient: 9.999999999621423


In [1]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._children = _children   # computation graph
        self._op = _op               # operation
        self._backward = lambda: None  # default empty function

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


    def __mul__(self, other):
        # Ensure both are Value objects
        other = other if isinstance(other, Value) else Value(other)
        
        # Create output node
        out = Value(self.data * other.data, (self, other), '*')

        # Backward function
        def _backward():
            # Gradient rule:
            # d/dself (self * other) = other
            # d/dother (self * other) = self
            # Apply chain rule with out.grad
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

### Declare Objects

In [2]:
v1 = Value(4)
v2 = 4   # auto-converted

### Forward Pass

In [3]:
result_mul = v1 * v2
print("Multiplication:", result_mul, "\nV1: ", v1, "\nV2: ",v2)

Multiplication: Value(data=16, grad=0) 
V1:  Value(data=4, grad=0) 
V2:  4


### Gradient Calculate only when backward pass

In [4]:
result_mul.grad = 1 # Seed gradient
result_mul._backward()
print("Multiplication:", result_mul, "\nV1: ", v1, "\nV2: ",v2)

Multiplication: Value(data=16, grad=1) 
V1:  Value(data=4, grad=4) 
V2:  4


## Seed Gradient in Backpropagation

```python
result_mul.grad = 1
```

### Why?

Because mathematically:

$$\frac{d(\text{output})}{d(\text{output})} = 1$$

This is the starting point of backpropagation.

### What should happen mathematically

If:

$$y = x \cdot 4$$

Then:

$$\frac{dy}{dx} = 4$$

### Chain Rule Explanation

Backpropagation applies the chain rule:

$$\frac{dy}{dx} = \frac{dy}{dy} \cdot \frac{d(x \cdot 4)}{dx}$$

$$= 1 \cdot 4 = 4$$

## Understanding Gradients

**real gradients in action**. Let’s interpret the output precisely.

### Our Output

```
Multiplication: Value(data=16, grad=1)
V1: Value(data=4, grad=8)
V2: 4
```

### What our computation is

```python
v1 = Value(4)
v2 = 4
result_mul = v1 * v2
```

Mathematically:

$$y = x \cdot 4$$

Where:
- x = v1 = 4
- y = result = 16

### Meaning of Each Gradient

####    **result_mul.grad = 1**

This is the seed gradient:

$$\frac{dy}{dy} = 1$$

Meaning:
- Output depends on itself fully
- So gradient = 1 (always)

#### **v1.grad = 8**

From our code:

```python
self.grad += other.data * out.grad
```

Substitute values:

```python
v1.grad += 4 * 1
```

Expected:

$$\frac{dy}{dx} = 4$$

we got:

```
v1.grad = 4
```

#### Important Rule

```python
self.grad += ...
```

Gradients are **accumulated**, not replaced.
#### Example:
```
First run → 4
Second run → 4 + 4 = 8
```

#### Intuition 

> Gradient = “How much output changes if input changes”

> v1.grad = 4 → Increasing x by 1 increases output by 4

> Gradients accumulate — if you don’t reset, they stack.

# Full Engine

In [ ]:
import math

class Value:

    def __init__(self, data, children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        
        out._backward = _backward
        return out
    
    def __mul__(self,other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward
        return out
